In [0]:
from pyspark.sql.functions import col, explode_outer, to_timestamp

BRONZE_TABLE = "gharchive_dev.v1_pyspark.gharchive_bronze"
SILVER_TABLE = "gharchive_dev.v1_pyspark.gharchive_silver"

# Read from bronze
df_bronze = spark.read.table(BRONZE_TABLE)

# Flatten repo struct
df = df_bronze
if "repo" in df.columns:
    df = (df
        .withColumn("repo_id", col("repo.id"))
        .withColumn("repo_name", col("repo.name"))
        .withColumn("repo_url", col("repo.url"))
    )

# Flatten actor struct
if "actor" in df.columns:
    df = (df
        .withColumn("actor_id", col("actor.id"))
        .withColumn("actor_login", col("actor.login"))
        .withColumn("actor_display_login", col("actor.display_login"))
        .withColumn("actor_avatar_url", col("actor.avatar_url"))
    )

# Explode payload.commits (if present) into separate rows
df = df.withColumn("commit", explode_outer("payload.commits"))

# Flatten commit struct
df = (df
    .withColumn("commit_sha", col("commit.sha"))
    .withColumn("commit_message", col("commit.message"))
    .withColumn("commit_author_name", col("commit.author.name"))
    .withColumn("commit_author_email", col("commit.author.email"))
)

# Parse created_at to timestamp
df = df.withColumn("created_at", to_timestamp("created_at"))

# Drop original nested columns
df_silver = df.drop("repo", "actor", "payload", "commit", "org")

# Write as Delta table
df_silver.write.format("delta").mode("overwrite").saveAsTable(SILVER_TABLE)

print(f"Silver table written: {SILVER_TABLE}")
print(f"Row count: {spark.read.table(SILVER_TABLE).count()}")